# 10 — Calidad, distribuciones y missingness

Auditoría agregada de las fuentes predictoras, cobertura por ventana, distribuciones de valores y completitud SOFA. Solo se resumen development y validation; el test permanece cerrado. Los motivos de exclusión de eventos pueden solaparse y por ello no deben sumarse como categorías mutuamente excluyentes.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, tempfile
import pandas as pd
from IPython.display import SVG, display
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src')) if str(PROJECT_ROOT / 'src') not in sys.path else None
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None
from mimic_sepsis.artifacts import ArtifactStore, ArtifactValidationError
from mimic_sepsis.descriptive import (feature_quality_summary, feature_source_quality_summary, feature_value_summary, prepare_descriptive_cohort, sofa_completeness_summary)
from mimic_sepsis.feature_sources import LAB_ITEMS, VITAL_ITEMS
from scripts.build_demo_sofa_incremental import canonical_config
config = canonical_config()
candidates = sorted((PROJECT_ROOT / 'data/derived/sofa').glob('*/60_features/sepsis3_development_features.manifest.json'))
valid = []
for manifest_path in candidates:
    store = ArtifactStore(manifest_path.parent)
    try:
        manifest = store.validate('sepsis3_development_features', expected_config=config)
        valid.append((manifest.created_at_utc, manifest_path.parents[1]))
    except (FileNotFoundError, ArtifactValidationError):
        pass
if not valid:
    raise RuntimeError('No hay artefactos compatibles. Ejecute: python scripts/build_demo_sofa_incremental.py --stage all --resume')
RUN_ROOT = sorted(valid, key=lambda item: (item[0], str(item[1])))[-1][1]
cohort_store = ArtifactStore(RUN_ROOT / '00_cohort'); score_store = ArtifactStore(RUN_ROOT / '30_score')
label_store = ArtifactStore(RUN_ROOT / '40_labels'); feature_store = ArtifactStore(RUN_ROOT / '60_features')
cohort = cohort_store.read_dataframe('cohort_stays', expected_config=config)
sepsis_stays = label_store.read_dataframe('sepsis_stays', expected_config=config)
shock_stays = label_store.read_dataframe('septic_shock_stays', expected_config=config)
clinical = prepare_descriptive_cohort(cohort, sepsis_stays, shock_stays, split_config=config['splits'])
features = {partition: feature_store.read_dataframe(f'sepsis3_{partition}_features', expected_config=config) for partition in ('development', 'validation')}
sofa_hourly = score_store.read_dataframe('sofa_hourly', expected_config=config)
variables = tuple(config['features']['variables'])
privacy_minimum = int(json.loads((PROJECT_ROOT / 'config/subgroups.json').read_text())['privacy_minimum_cell'])
print(f'Ejecución validada: {RUN_ROOT.name}')

## Reglas y pérdidas desde las fuentes

El inventario fija itemids, límites y conversiones. El resumen posterior cuenta candidatos y motivos de exclusión solo en las estancias development/validation. Un laboratorio con `storetime` anterior a la muestra se rechaza para impedir backdating.

In [ ]:
rule_rows = []
for source, mapping in [('chartevents', VITAL_ITEMS), ('labevents', LAB_ITEMS)]:
    for itemid, (variable, lower, upper) in mapping.items():
        rule_rows.append({'source': source, 'itemid': itemid, 'variable': 'temperature' if variable == 'temperature_f' else variable, 'raw_lower': lower, 'raw_upper': upper, 'transformation': 'F_to_C' if variable == 'temperature_f' else 'identity'})
rule_inventory = pd.DataFrame(rule_rows)
data_dir = PROJECT_ROOT / 'data/mimic-iv-demo/2.2'
chartevents = pd.read_csv(data_dir/'icu/chartevents.csv.gz', usecols=['stay_id','itemid','charttime','valuenum'])
labevents = pd.read_csv(data_dir/'hosp/labevents.csv.gz', usecols=['subject_id','hadm_id','itemid','charttime','storetime','valuenum'])
permitted_stays = set(clinical['stay_id']); permitted_admissions = set(zip(clinical['subject_id'], clinical['hadm_id']))
chartevents = chartevents.loc[chartevents['stay_id'].isin(permitted_stays)]
labevents = labevents.loc[[pair in permitted_admissions for pair in zip(labevents['subject_id'], labevents['hadm_id'])]]
source_quality = feature_source_quality_summary(chartevents, labevents, clinical, privacy_minimum_cell=privacy_minimum)
display(rule_inventory, source_quality)
del chartevents, labevents

## Cobertura, densidad y distribuciones

La cobertura se calcula en `[L-W,L)` para 6, 12 y 24 horas. Las distribuciones utilizan el último valor disponible en 24 horas; no se aplican winsorización ni imputación en este descriptivo.

In [ ]:
quality = feature_quality_summary(features, variables, privacy_minimum_cell=privacy_minimum)
values = feature_value_summary(features, variables, privacy_minimum_cell=privacy_minimum)
sofa_coverage = sofa_completeness_summary(sofa_hourly, clinical, privacy_minimum_cell=privacy_minimum)
display(quality, values, sofa_coverage)
table_dir = PROJECT_ROOT / 'data/derived/reports/tables'; table_dir.mkdir(parents=True, exist_ok=True)
for name, table in {'10_source_rules': rule_inventory, '10_source_quality': source_quality, '10_feature_quality': quality, '10_value_distributions': values, '10_sofa_completeness': sofa_coverage}.items():
    table.to_csv(table_dir / f'{name}.csv', index=False)

## Figuras de calidad con ggplot2

In [ ]:
rscript = shutil.which('Rscript')
if not rscript: raise RuntimeError('Rscript no está disponible en el entorno.')
figure_dir = PROJECT_ROOT / 'data/derived/reports/figures'; figure_dir.mkdir(parents=True, exist_ok=True)
q_plot = quality.query('privacy_suppressed == False').copy(); v_plot = values.query('privacy_suppressed == False').copy(); s_plot = sofa_coverage.query('privacy_suppressed == False').copy()
r_code = r'''
args <- commandArgs(trailingOnly=TRUE)
suppressPackageStartupMessages(library(ggplot2))
q <- read.csv(args[1]); v <- read.csv(args[2]); s <- read.csv(args[3])
p1 <- ggplot(q, aes(factor(window_hours), variable, fill=percent_observed)) + geom_tile() + facet_wrap(~partition) + scale_fill_viridis_c(limits=c(0,100)) + labs(title='Cobertura de predictores por ventana', x='Lookback (h)', y=NULL, fill='% observado') + theme_minimal(base_size=10)
p2 <- ggplot(v, aes(median, reorder(variable, median), colour=partition, xmin=q1, xmax=q3)) + geom_errorbarh(height=.2, position=position_dodge(width=.45)) + geom_point(position=position_dodge(width=.45)) + labs(title='Último valor disponible en 24 h', subtitle='Mediana e IQR; escalas clínicas distintas', x='Valor', y=NULL, colour='Partición') + theme_minimal(base_size=10)
p3 <- ggplot(s, aes(percent_observed, reorder(component, percent_observed), fill=partition)) + geom_col(position='dodge') + scale_x_continuous(limits=c(0,100)) + labs(title='Completitud horaria de componentes SOFA', x='Horas observadas (%)', y=NULL, fill='Partición') + theme_minimal(base_size=10)
ggsave(args[4], p1, width=9, height=5.5, device=grDevices::svg)
ggsave(args[5], p2, width=9, height=5.5, device=grDevices::svg)
ggsave(args[6], p3, width=9, height=5.5, device=grDevices::svg)
'''
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp); q_csv = tmp/'quality.csv'; v_csv = tmp/'values.csv'; s_csv = tmp/'sofa.csv'
    q_plot.to_csv(q_csv, index=False); v_plot.to_csv(v_csv, index=False); s_plot.to_csv(s_csv, index=False)
    figures = [figure_dir/'10_feature_coverage.svg', figure_dir/'10_value_distributions.svg', figure_dir/'10_sofa_completeness.svg']
    run = subprocess.run([rscript, '-e', r_code, str(q_csv), str(v_csv), str(s_csv), *map(str, figures)], capture_output=True, text=True)
    if run.returncode: raise RuntimeError(run.stderr)
display(*(SVG(filename=str(path)) for path in figures))

## Criterio para avanzar

La calidad se considera técnicamente auditable cuando cada variable conserva reglas explícitas, los laboratorios nunca se adelantan a la muestra, la cobertura se presenta por partición/ventana y la ausencia permanece visible. El demo no permite fijar exclusiones por cobertura ni elegir transformaciones; esas decisiones requieren development completo y las firmas D011/D014.